# Baselines

Baselines against which QLoRA adapters are compared:
- **B1 — Zero-shot (base):** base model, artist name in prompt, no fine-tuning
- **B2 — Few-shot (base):** base model, artist name + 3 example lyrics
- **B3 — Zero-shot (instruct):** `-it` model, named band + genre, no examples
- **B4 — Few-shot (instruct):** `-it` model, named band + genre + 3 examples

B1/B2 run on the non-instruction-tuned base model (it can't follow "write in the style of X" → emits web text). B3/B4 run on the `-it` model with chat-formatted, realistic prompts. All four are computed + cached by `evaluate.py`.

In [1]:
import json

import pandas as pd

from config import ARTISTS, RESULTS_DIR

# Display-only: baselines are computed + cached by evaluate.py.
# Run `uv run python evaluate.py` first.
BASELINES_DIR = RESULTS_DIR / "baselines"


def load_baseline(artist, method):
    f = BASELINES_DIR / artist.lower().replace(" ", "_") / f"{method}.json"
    if not f.exists():
        return None
    hit = json.load(open(f))
    return {"samples": hit["samples"], "df": pd.DataFrame(hit["df"])}

## B1 -- Zero-shot

Artist name in prompt, no examples, no fine-tuning.

In [2]:
zero_shot = {}
for artist in ARTISTS:
    hit = load_baseline(artist, "zero_shot")
    if hit is None:
        print(f"missing {artist} zero-shot -- run evaluate.py")
        continue
    zero_shot[artist] = hit
    df = hit["df"]
    print(f"{artist}: {df[artist].mean():.4f} +/- {df[artist].std():.4f}")

Gojira: 0.0017 +/- 0.0006
Tool: 0.9927 +/- 0.0030
Death: 0.0016 +/- 0.0023
Mastodon: 0.0013 +/- 0.0003
Opeth: 0.0015 +/- 0.0004


## B2 -- Few-shot (3 examples)

Artist name + 3 training lyrics as in-context examples.

In [3]:
few_shot = {}
for artist in ARTISTS:
    hit = load_baseline(artist, "few_shot")
    if hit is None:
        print(f"missing {artist} few-shot -- run evaluate.py")
        continue
    few_shot[artist] = hit
    df = hit["df"]
    print(f"{artist}: {df[artist].mean():.4f} +/- {df[artist].std():.4f}")

Gojira: 0.0061 +/- 0.0108
Tool: 0.7846 +/- 0.3975
Death: 0.0016 +/- 0.0013
Mastodon: 0.0061 +/- 0.0108
Opeth: 0.0035 +/- 0.0050


## B3 — Zero-shot (instruct)

Instruction-tuned model, named band + genre, no examples.

In [4]:
zero_shot_it = {}
for artist in ARTISTS:
    hit = load_baseline(artist, "zero_shot_it")
    if hit is None:
        print(f"missing {artist} zero-shot (it) -- run evaluate.py")
        continue
    zero_shot_it[artist] = hit
    df = hit["df"]
    print(f"{artist}: {df[artist].mean():.4f} +/- {df[artist].std():.4f}")

Gojira: 0.0102 +/- 0.0185
Tool: 0.0030 +/- 0.0016
Death: 0.1005 +/- 0.3140
Mastodon: 0.0071 +/- 0.0090
Opeth: 0.9955 +/- 0.0003


## B4 — Few-shot (instruct, 3 examples)

Instruction-tuned model, named band + genre + 3 in-context examples.

In [5]:
few_shot_it = {}
for artist in ARTISTS:
    hit = load_baseline(artist, "few_shot_it")
    if hit is None:
        print(f"missing {artist} few-shot (it) -- run evaluate.py")
        continue
    few_shot_it[artist] = hit
    df = hit["df"]
    print(f"{artist}: {df[artist].mean():.4f} +/- {df[artist].std():.4f}")

Gojira: 0.0112 +/- 0.0293
Tool: 0.0537 +/- 0.1557
Death: 0.0011 +/- 0.0001
Mastodon: 0.0089 +/- 0.0209
Opeth: 0.9958 +/- 0.0002


## Summary

In [6]:
methods = [("zero_shot", "Zero-shot"), ("few_shot", "Few-shot"),
           ("zero_shot_it", "Zero-shot (it)"), ("few_shot_it", "Few-shot (it)")]
loaded = {"zero_shot": zero_shot, "few_shot": few_shot,
          "zero_shot_it": zero_shot_it, "few_shot_it": few_shot_it}

print("Target-artist mean attribution (higher = better):\n")
header = f"{'Artist':<12}" + "".join(f"{lbl:>16}" for _, lbl in methods)
print(header)
print("-" * len(header))
for artist in ARTISTS:
    cells = []
    for key, _ in methods:
        store = loaded[key]
        v = store[artist]["df"][artist].mean() if artist in store else float("nan")
        cells.append(f"{v:>16.4f}")
    print(f"{artist:<12}" + "".join(cells))

Target-artist mean attribution (higher = better):

Artist             Zero-shot        Few-shot  Zero-shot (it)   Few-shot (it)
----------------------------------------------------------------------------
Gojira                0.0017          0.0061          0.0102          0.0112
Tool                  0.9927          0.7846          0.0030          0.0537
Death                 0.0016          0.0016          0.1005          0.0011
Mastodon              0.0013          0.0061          0.0071          0.0089
Opeth                 0.0015          0.0035          0.9955          0.9958
